In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16)
model = model.to("cuda")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

c:\Projects\nlp-toolkit\hf-toolkit-env\lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Pratham\.cache\huggingface\hub\models--Qwen--Qwen2.5-1.5B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [2]:
messages = [
    {"role": "user", "content": "Explain what a transformer model is in 2 simple sentences."}
]

prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
print(prompt)

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
Explain what a transformer model is in 2 simple sentences.<|im_end|>
<|im_start|>assistant



In [3]:
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=True,
        temperature=0.7,
        top_p=0.9
    )

response = tokenizer.decode(output_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print(response)

A transformer model is a type of deep learning architecture that uses self-attention mechanisms to process and analyze sequential data efficiently. It was developed by Google Brain researchers and has been widely used for tasks such as natural language processing, speech recognition, and computer vision.


In [4]:
def generate_response(do_sample, temperature=0.7, top_p=0.9):
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=60,
            do_sample=do_sample,
            temperature=temperature if do_sample else None,
            top_p=top_p if do_sample else None,
        )
    return tokenizer.decode(output_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

print("Greedy (do_sample=False):")
print(generate_response(do_sample=False))
print("\nSampled, temperature=0.3 (low randomness):")
print(generate_response(do_sample=True, temperature=0.3))
print("\nSampled, temperature=1.2 (high randomness):")
print(generate_response(do_sample=True, temperature=1.2))

Greedy (do_sample=False):
A transformer model is a type of neural network architecture designed to process and understand sequential data like text or audio, using self-attention mechanisms to efficiently capture dependencies between elements in the sequence.

Sampled, temperature=0.3 (low randomness):
A transformer model is a type of deep learning architecture designed to process and understand sequential data like text or speech. It uses self-attention mechanisms to focus on relevant parts of the input sequence for each position, allowing it to capture long-range dependencies effectively.

Sampled, temperature=1.2 (high randomness):
A transformer model is an advanced machine learning architecture that uses self-attention mechanisms to process input data and generate meaningful representations for natural language or other sequential inputs. It outperforms earlier models on various NLP tasks due to its ability to focus on relevant parts of the input independently.


In [5]:
from transformers import TextIteratorStreamer
from threading import Thread

streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)

generation_kwargs = dict(
    **inputs,
    max_new_tokens=100,
    do_sample=True,
    temperature=0.7,
    top_p=0.9,
    streamer=streamer
)

thread = Thread(target=model.generate, kwargs=generation_kwargs)
thread.start()

for new_text in streamer:
    print(new_text, end="", flush=True)

A transformer model is a type of deep learning architecture designed to process and understand sequential data like text or audio efficiently. It uses self-attention mechanisms to focus on relevant parts of the input for processing instead of treating all inputs equally.